# EXP-005D — Deterministic EXP-005A Replay Control

Control experiment after EXP-005C regression. This notebook preserves the EXP-005A source-BFS + hidden-state-hash backbone, removes EXP-005B broad scan and EXP-005C transfer, and replaces wall-clock random seeding with deterministic game/level seeding.

Current reference: EXP-005A Version 9 public score `0.17`. EXP-005D Version 13 scored `0.10`, so the deterministic control did not reproduce EXP-005A.

Post-result update: the agent now writes scoring-time attribution diagnostics for source lookup/load, BFS solve status, BFS replay counts, fallback reasons, and deterministic level seeds.

Expected artifacts: `/kaggle/working/exp005d_bfs_events.jsonl` and `/kaggle/working/exp005d_run_summary.json`.


In [ ]:
!pip install -q --no-index --find-links \
    /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels \
    arc-agi python-dotenv



In [ ]:
%%writefile /kaggle/working/my_agent.py
import copy, glob, hashlib, importlib.util, json, os, random, time, traceback
from collections import Counter, deque
from pathlib import Path
import numpy as np
from agents.agent import Agent
from arcengine import ActionInput, GameAction, GameState

VARIANT='EXP-005D'
BFS_EVENT_PATH=Path('/kaggle/working/exp005d_bfs_events.jsonl')
RUN_SUMMARY_PATH=Path('/kaggle/working/exp005d_run_summary.json')

def stable_seed(*parts):
    txt='|'.join(str(p) for p in parts)
    return int(hashlib.md5(txt.encode('utf-8')).hexdigest()[:8],16)
def logj(x):
    try:
        BFS_EVENT_PATH.parent.mkdir(parents=True,exist_ok=True)
        BFS_EVENT_PATH.open('a').write(json.dumps(x,default=str)+'\n')
    except Exception: pass
def savej(x):
    try: RUN_SUMMARY_PATH.write_text(json.dumps(x,indent=2,default=str))
    except Exception: pass
def aid(a):
    try: return int(a.value)
    except Exception: return int(a)
def lf(obj):
    fr=getattr(obj,'frame',None)
    if fr is None: return None
    ar=np.asarray(fr)
    return ar[-1].astype(np.int64) if ar.ndim==3 else ar.astype(np.int64) if ar.ndim==2 else None
def ai(i,data=None):
    g=GameAction.from_id(int(i)); return ActionInput(id=g,data=dict(data)) if data else ActionInput(id=g)
def act(i,data=None):
    a=GameAction.from_id(int(i))
    if data: a.set_data(dict(data))
    return a

def find_src(game_id,arc_env=None):
    gid=str(game_id).split('-')[0]
    guess=(gid[0].upper()+gid[1:]) if gid and gid[0].isalpha() else gid.capitalize()
    def cls(path):
        try:
            import re
            txt=Path(path).read_text(errors='ignore')[:5000]
            m=re.search(r'class\s+(\w+)\s*\(\s*ARCBaseGame',txt)
            return m.group(1) if m else guess
        except Exception: return guess
    try:
        d=getattr(getattr(arc_env,'environment_info',None),'local_dir',None)
        if d:
            r=Path(d)
            c=[r/f'{gid}.py',r/f'{gid.lower()}.py',r/f'{guess.lower()}.py']+list(r.rglob(f'{gid}.py'))[:20]+list(r.rglob(f'{gid.lower()}.py'))[:20]
            for p in c:
                if p.exists(): return str(p),cls(p)
    except Exception: pass
    for pat in [f'/tmp/**/{gid}.py',f'/kaggle/working/**/{gid}.py',f'/kaggle/input/**/{gid}.py',f'**/game_sources/**/{gid}.py']:
        try:
            m=glob.glob(pat,recursive=True)
            if m: return m[0],cls(m[0])
        except Exception: pass
    return None,guess

class Solver:
    def __init__(s,path,cls_name,game_id='unknown'):
        s.path=path; s.cls_name=cls_name; s.game_id=game_id; s.game_cls=None; s.last_log={}
        s.scan_timeout=float(os.getenv('EXP005D_SCAN_TIMEOUT','4'))
        s.bfs_timeout=float(os.getenv('EXP005D_BFS_TIMEOUT','25'))
        s.max_states=int(os.getenv('EXP005D_MAX_STATES','50000'))
        s.max_depth=int(os.getenv('EXP005D_MAX_DEPTH','30'))
    def load(s):
        try:
            spec=importlib.util.spec_from_file_location('exp005d_game',s.path)
            mod=importlib.util.module_from_spec(spec); spec.loader.exec_module(mod)
            s.game_cls=getattr(mod,s.cls_name)
            logj({'event':'load','variant':VARIANT,'status':'ok','game_id':s.game_id,'path':s.path,'class':s.cls_name})
            return True
        except Exception as e:
            s.last_log={'event':'load','variant':VARIANT,'status':'error','game_id':s.game_id,'path':s.path,'class':s.cls_name,'error':repr(e)}; logj(s.last_log); return False
    def reset(s,lvl):
        g=s.game_cls(); g.set_level(int(lvl))
        g.perform_action(ai(GameAction.RESET.value),raw=True)
        r=g.perform_action(ai(GameAction.RESET.value),raw=True); f=lf(r)
        if f is None: f=np.asarray(g.get_pixels(0,0,64,64),dtype=np.int64)
        return g,f
    def fh(s,f): return hashlib.md5(np.asarray(f,dtype=np.uint8).tobytes()).hexdigest()
    def scalars(s,g):
        ex={'_action_count','_full_reset','_action_complete','_last_action','_last_frame','_rng','rng','np_random'}; out={}
        for k,v in getattr(g,'__dict__',{}).items():
            if k in ex or k.startswith('__'): continue
            if isinstance(v,(int,float,bool,str)) and (not isinstance(v,str) or len(v)<=80): out[k]=v
        return out
    def sh(s,g,f,hid):
        h=s.fh(f); vals=[]
        for k in hid:
            try: vals.append(f'{k}={getattr(g,k,None)}')
            except Exception: pass
        return h+'|'+'|'.join(vals) if vals else h
    def av(s,g):
        out=[]
        for a in getattr(g,'_available_actions',[]):
            try:
                x=aid(a)
                if x not in out: out.append(x)
            except Exception: pass
        return out
    def scan(s,g0,f0):
        acts=[]; bg=int(np.bincount(f0.flatten(),minlength=16).argmax()); ids=s.av(g0)
        stats={'available':ids,'dir':0,'click':0,'dedup':0,'total':0,'scan_style':'exp005a_simple_stride2'}
        for a in [x for x in ids if 1<=x<=5]:
            try:
                g=copy.deepcopy(g0); r=g.perform_action(ai(a),raw=True); f=lf(r)
                if f is not None and np.sum(f0!=f)>0: acts.append((a,None)); stats['dir']+=1
            except Exception: pass
        if 6 in ids:
            t=time.time(); seen=set()
            for y in range(0,64,2):
                if time.time()-t>s.scan_timeout: break
                for x in range(0,64,2):
                    if f0[y,x]==bg: continue
                    data={'x':int(x),'y':int(y),'game_id':'exp005d_bfs'}
                    try:
                        g=copy.deepcopy(g0); r=g.perform_action(ai(6,data),raw=True); f=lf(r)
                        if f is not None and np.sum(f0!=f)>0:
                            eh=s.fh(f)
                            if eh not in seen: seen.add(eh); acts.append((6,data)); stats['click']+=1
                            else: stats['dedup']+=1
                    except Exception: pass
        stats['total']=len(acts); return acts,stats
    def probe(s,g0,acts):
        base=s.scalars(g0); ch=set()
        for a,d in acts[:12]:
            try:
                g=copy.deepcopy(g0); g.perform_action(ai(a,d),raw=True); aft=s.scalars(g)
                for k,v in base.items():
                    if k in aft and aft[k]!=v: ch.add(k)
            except Exception: pass
        return [k for k in sorted(ch) if not any(n in k.lower() for n in ('time','timestamp','elapsed','duration'))]
    def solve(s,lvl):
        t=time.time(); log={'event':'solve','variant':VARIANT,'game_id':s.game_id,'level':int(lvl),'path':s.path,'class':s.cls_name,'status':'started','scan':{},'hidden':[],'explored':0,'unique':0,'solution_len':None}
        try:
            base,f0=s.reset(lvl); acts,st=s.scan(base,f0); hid=s.probe(base,acts)
            log.update({'scan':st,'hidden':hid,'hidden_hash':bool(hid)})
            if not acts: log['status']='no_effective_actions'; log['elapsed']=round(time.time()-t,3); s.last_log=log; logj(log); return None
            q=deque([(copy.deepcopy(base),[],0)]); seen={s.sh(base,f0,hid)}
            while q and log['explored']<s.max_states and time.time()-t<s.bfs_timeout:
                g,hist,depth=q.popleft()
                for a,d in acts:
                    if log['explored']>=s.max_states or time.time()-t>=s.bfs_timeout: break
                    try:
                        g2=copy.deepcopy(g); r=g2.perform_action(ai(a,d),raw=True); log['explored']+=1; f=lf(r)
                        if f is None: continue
                        hh=s.sh(g2,f,hid)
                        if hh in seen: continue
                        seen.add(hh); nh=hist+[(a,d)]
                        cur=getattr(g2,'_current_level_index',lvl); done=getattr(r,'levels_completed',lvl)
                        if done>lvl or cur>lvl:
                            log.update({'unique':len(seen),'solution_len':len(nh),'status':'solved','elapsed':round(time.time()-t,3)}); s.last_log=log; logj(log); return nh
                        if depth+1<s.max_depth: q.append((g2,nh,depth+1))
                    except Exception: pass
            log.update({'unique':len(seen),'status':'timeout_or_exhausted','elapsed':round(time.time()-t,3)}); s.last_log=log; logj(log); return None
        except Exception as e:
            log.update({'status':'error','error':repr(e),'elapsed':round(time.time()-t,3)}); s.last_log=log; logj(log); return None

class MyAgent(Agent):
    MAX_ACTIONS=float('inf'); _MAX_FRAMES=10
    def __init__(s,*a,**kw):
        super().__init__(*a,**kw)
        s.base_seed=stable_seed(VARIANT,getattr(s,'game_id','unknown'))
        s.rng=random.Random(s.base_seed); np.random.seed(s.base_seed%(2**32-1))
        s.current_level=None; s.bfs=None; s.bfs_ready=False; s.src=None; s.cls=None; s.solution=None; s.step=0; s.tried=set(); s.policy=Counter(); s.actions=Counter(); s.logs=[]; s.start=time.time()
        s.attribution=Counter(); s.fallback_reasons=Counter(); s.last_bfs_status={}; s.last_fallback_reason=None; s.last_level_seed=None
    def summary(s):
        savej({'variant':VARIANT,'game_id':getattr(s,'game_id',None),'deterministic_seed':s.base_seed,'last_level_seed':s.last_level_seed,'source':s.src,'class':s.cls,'levels':sorted(list(s.tried)),'policy':dict(s.policy),'actions':dict(s.actions),'attribution':dict(s.attribution),'fallback_reasons':dict(s.fallback_reasons),'last_bfs_status':s.last_bfs_status,'last_fallback_reason':s.last_fallback_reason,'logs':s.logs[-40:]})
    def append_frame(s,f):
        s.frames.append(f); s.frames=s.frames[-s._MAX_FRAMES:]
        if getattr(f,'guid',None): s.guid=f.guid
        if hasattr(s,'recorder') and not s.is_playback:
            try: s.recorder.record(json.loads(f.model_dump_json()))
            except Exception: pass
    def level(s,lfx): return int(getattr(lfx,'levels_completed',0) or 0)
    def raw(s,lfx):
        f=lf(lfx); return f if f is not None else np.zeros((64,64),dtype=np.int64)
    def avail(s,lfx):
        out=[]
        for a in getattr(lfx,'available_actions',None) or []:
            try:
                x=aid(a)
                if x not in out: out.append(x)
            except Exception: pass
        return out
    def init_bfs(s):
        if s.bfs_ready: return
        s.src,s.cls=find_src(s.game_id,getattr(s,'arc_env',None))
        lookup={'event':'source_lookup','variant':VARIANT,'status':'found' if s.src else 'missing','game_id':getattr(s,'game_id',None),'source':s.src,'class':s.cls}
        s.logs.append(lookup); logj(lookup)
        if s.src:
            s.bfs=Solver(s.src,s.cls,game_id=getattr(s,'game_id','unknown'))
            if not s.bfs.load(): s.bfs=None
        if s.bfs is None:
            p={'event':'bfs_init','variant':VARIANT,'status':'source_not_found_or_load_failed','game_id':getattr(s,'game_id',None),'source':s.src,'class':s.cls}; s.logs.append(p); logj(p); s.attribution['bfs_unavailable']+=1
        s.bfs_ready=True; s.summary()
    def try_bfs(s,lvl):
        if lvl in s.tried: return
        s.tried.add(lvl); s.init_bfs()
        if s.bfs is None:
            s.last_bfs_status[str(lvl)]='source_not_found_or_load_failed'; s.summary(); return
        sol=s.bfs.solve(lvl); status=str(s.bfs.last_log.get('status','unknown')); s.last_bfs_status[str(lvl)]=status; s.logs.append(dict(s.bfs.last_log))
        if sol:
            s.solution=sol; s.step=0; s.attribution['bfs_solved_levels']+=1
        else:
            s.attribution['bfs_no_solution_levels']+=1; s.attribution[f'bfs_status:{status}']+=1
        s.summary()
    def reseed_level(s,lvl):
        seed=stable_seed(VARIANT,getattr(s,'game_id','unknown'),'level',int(lvl))
        s.rng.seed(seed); np.random.seed(seed%(2**32-1)); s.last_level_seed=seed; return seed
    def click(s,raw):
        bg=int(np.bincount(raw.flatten(),minlength=16).argmax()); ys,xs=np.where(raw!=bg)
        if len(xs)==0: return None
        i=s.rng.randrange(len(xs)); return {'x':int(xs[i]),'y':int(ys[i]),'game_id':'exp005d_fallback'}
    def fallback(s,raw,lfx,reason='bfs_no_solution'):
        s.last_fallback_reason=reason; s.fallback_reasons[reason]+=1; s.attribution['fallback_actions']+=1
        av=[x for x in s.avail(lfx) if x in [1,2,3,4,5,6,7]]
        if 6 in av and s.rng.random()<0.65:
            c=s.click(raw)
            if c:
                a=GameAction.ACTION6; a.set_data(c); a.reasoning='fallback:visible_click:deterministic'; return a,6
        mv=[x for x in av if 1<=x<=5]
        if mv:
            x=s.rng.choice(mv); a=act(x); a.reasoning='fallback:random_move:deterministic'; return a,x
        if 7 in av:
            a=act(7); a.reasoning='fallback:undo_last_resort'; return a,7
        a=GameAction.RESET; a.reasoning='fallback:reset_last_resort'; return a,0
    def is_done(s,frames,lfx):
        try:
            d=lfx.state is GameState.WIN or time.time()-s.start>=8*3600-300
            if d: s.summary()
            return d
        except Exception: s.summary(); return True
    def choose_action(s,frames,lfx):
        try:
            lvl=s.level(lfx); raw=s.raw(lfx); st=getattr(lfx,'state',None)
            if st in [GameState.NOT_PLAYED,GameState.GAME_OVER]: s.policy['reset']+=1; s.attribution['reset_actions']+=1; a=GameAction.RESET; a.reasoning='reset'; return a
            if lvl!=s.current_level:
                s.current_level=lvl; s.solution=None; s.step=0; s.reseed_level(lvl); s.try_bfs(lvl)
            if s.solution and s.step<len(s.solution):
                i,d=s.solution[s.step]; s.step+=1; a=act(i,d); a.reasoning=f'bfs_replay:{s.step}/{len(s.solution)}'; s.policy['bfs_replay']+=1; s.attribution['bfs_replay_actions']+=1; s.actions[int(i)]+=1; s.summary(); return a
            reason='bfs_solution_exhausted' if s.solution else 'bfs_'+str(s.last_bfs_status.get(str(lvl),'not_attempted'))
            a,i=s.fallback(raw,lfx,reason=reason); s.policy['fallback']+=1; s.actions[int(i)]+=1; return a
        except Exception as e:
            traceback.print_exc(); s.policy['error_fallback']+=1; s.attribution['error_fallback_actions']+=1; s.summary(); x=s.rng.choice([1,2,3,4,5]); a=act(x); a.reasoning=f'error_fallback:deterministic:{e}'; return a



In [ ]:
import os, shutil, subprocess
from pathlib import Path
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    BASE=Path('/kaggle/working/ARC-AGI-3-Agents'); SRC=Path('/kaggle/input/competitions/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents')
    AGENT_SRC=Path('/kaggle/working/my_agent.py'); AGENT_DST=BASE/'agents/templates/my_agent.py'
    subprocess.run(['curl','--fail','--retry','60','--retry-all-errors','--retry-delay','5','--retry-max-time','300','http://gateway:8001/api/games'], check=False)
    if BASE.exists(): shutil.rmtree(BASE)
    shutil.copytree(SRC,BASE); shutil.copy(AGENT_SRC,AGENT_DST)
    (BASE/'agents/__init__.py').write_text('from typing import Type\nfrom dotenv import load_dotenv\nfrom .agent import Agent, Playback\nfrom .swarm import Swarm\nfrom .templates.random_agent import Random\nfrom .templates.my_agent import MyAgent\nload_dotenv()\nAVAILABLE_AGENTS: dict[str, Type[Agent]] = {"random": Random, "myagent": MyAgent}\n')
    (BASE/'.env').write_text('\n'.join(['SCHEME=http','HOST=gateway','PORT=8001','ARC_API_KEY=test-key-123','ARC_BASE_URL=http://gateway:8001/','OPERATION_MODE=online','RECORDINGS_DIR=/kaggle/working/server_recording'])+'\n')
    subprocess.run(['python','main.py','--agent','myagent'],cwd=str(BASE),env={**os.environ,'MPLBACKEND':'agg','PYTHONUNBUFFERED':'1'},check=True)
else:
    print('Local/non-rerun mode: writing dummy submission fallback.')



In [ ]:
import os
if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    import pandas as pd
    pd.DataFrame([{'row_id':'debug_0','game_id':'1','end_of_game':True,'score':0}]).to_parquet('/kaggle/working/submission.parquet',index=False)
    print('Wrote /kaggle/working/submission.parquet dummy fallback.')



In [ ]:
from pathlib import Path
import json, os
BFS_EVENT_PATH=Path('/kaggle/working/exp005d_bfs_events.jsonl')
RUN_SUMMARY_PATH=Path('/kaggle/working/exp005d_run_summary.json')
if not BFS_EVENT_PATH.exists():
    BFS_EVENT_PATH.write_text(json.dumps({'variant':'EXP-005D','event':'artifact_placeholder','mode':'non_rerun_notebook_mode','status':'placeholder_only','reason':'KAGGLE_IS_COMPETITION_RERUN was not set, so main.py --agent myagent did not run.','expected_real_artifact_when_scored':str(BFS_EVENT_PATH)})+'\n')
if not RUN_SUMMARY_PATH.exists():
    RUN_SUMMARY_PATH.write_text(json.dumps({'variant':'EXP-005D','mode':'non_rerun_notebook_mode','status':'placeholder_only','purpose':'deterministic EXP-005A replay control','current_best_reference':{'experiment':'EXP-005A','public_score':0.17,'version':9},'recent_regressions':{'EXP-005B':0.09,'EXP-005C_V12':0.10},'expected_real_fields_when_scored':['source','class','levels','policy','actions','attribution','fallback_reasons','last_bfs_status','last_fallback_reason','logs','hidden','hidden_hash','scan','status','solution_len','deterministic_seed','last_level_seed']},indent=2))
print('EXP-005D artifact check:')
print(' -',BFS_EVENT_PATH,'exists:',BFS_EVENT_PATH.exists(),'size:',BFS_EVENT_PATH.stat().st_size)
print(' -',RUN_SUMMARY_PATH,'exists:',RUN_SUMMARY_PATH.exists(),'size:',RUN_SUMMARY_PATH.stat().st_size)



## Tracker row draft

```text
| EXP-005D | 2026-05-22/23 | `notebooks/01_exploration/exp005d_deterministic_exp005a_control.ipynb` | Deterministic EXP-005A replay control + attribution diagnostics | EXP-005A backbone with hidden-state hash and simple scan; wall-clock seed replaced by deterministic game/level seed; no EXP-005B broad scan; no EXP-005C transfer; post-V13 update adds BFS/fallback attribution fields | Visible notebook artifacts are placeholder-only | Public score: 0.10 V13; diagnostic rerun pending | Regression / attribution update ready | Goal: isolate whether fallback was caused by source load, scan/no-actions, BFS timeout/exhaustion, BFS error, or no solution. |
```
